# 7.4 Performance Tuning — Apply Notebook

## Objective

Systematically tune ONNX Runtime inference performance by profiling,
sweeping thread counts, comparing optimisation levels, and building
an auto-tuner.

| # | Exercise | Key Skill |
|---|----------|-----------|
| 1 | Profiling ORT sessions | `enable_profiling`, Chrome-trace |
| 2 | Thread count sweep (intra-op / inter-op) | `intra_op_num_threads`, `inter_op_num_threads` |
| 3 | Graph optimisation level comparison | DISABLED → BASIC → EXTENDED → ALL |
| 4 | Memory arena configuration | `enable_cpu_mem_arena`, `enable_mem_pattern` |
| 5 | Batch size vs latency analysis | Throughput-latency trade-off |
| 6 | Full benchmark with percentile statistics | p50, p95, p99, std |
| 7 | Model warm-up best practices | Cold-start elimination |
| 8 | **Challenge:** build an auto-tuner | Grid search for optimal settings |

```
pip install onnx onnxruntime numpy
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import os, sys, time, json, tempfile, warnings
warnings.filterwarnings("ignore")

import numpy as np

import onnx
from onnx import checker, helper, TensorProto, numpy_helper

import onnxruntime as ort

print(f"ONNX         : {onnx.__version__}")
print(f"ORT          : {ort.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"CPU count    : {os.cpu_count()}")
print(f"Providers    : {ort.get_available_providers()}")

WORK_DIR = tempfile.mkdtemp(prefix="ort_perf_")
print(f"Working dir  : {WORK_DIR}")

### Shared Model: Medium MLP

We build a model large enough to make performance differences measurable:

$$Y = \text{ReLU}(\text{ReLU}(\text{ReLU}(XW_1 + B_1) W_2 + B_2) W_3 + B_3)$$

Three layers: $128 \to 256 \to 128 \to 64$.

In [ ]:
def build_mlp(path, layers=((128, 256), (256, 128), (128, 64)), opset=17):
    """Build a multi-layer ReLU network."""
    rng = np.random.default_rng(0)
    nodes, initializers = [], []
    prev_name = "X"

    for i, (inf, outf) in enumerate(layers):
        w_name, b_name = f"W{i}", f"B{i}"
        mm_name, add_name, relu_name = f"mm{i}", f"add{i}", f"relu{i}"

        W = rng.standard_normal((inf, outf)).astype(np.float32) * 0.1
        B = rng.standard_normal((outf,)).astype(np.float32) * 0.1
        initializers.extend([
            numpy_helper.from_array(W, name=w_name),
            numpy_helper.from_array(B, name=b_name),
        ])
        nodes.extend([
            helper.make_node("MatMul", [prev_name, w_name], [mm_name]),
            helper.make_node("Add", [mm_name, b_name], [add_name]),
            helper.make_node("Relu", [add_name], [relu_name]),
        ])
        prev_name = relu_name

    in_f = layers[0][0]
    out_f = layers[-1][1]
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["N", in_f])
    Y = helper.make_tensor_value_info(prev_name, TensorProto.FLOAT, ["N", out_f])

    graph = helper.make_graph(nodes, "MLP", [X], [Y], initializer=initializers)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])
    checker.check_model(model)
    onnx.save(model, path)
    return model


MODEL_PATH = os.path.join(WORK_DIR, "mlp.onnx")
build_mlp(MODEL_PATH)

proto = onnx.load(MODEL_PATH)
n_params = sum(int(np.prod(i.dims)) for i in proto.graph.initializer)
print(f"Model: 3-layer MLP (128→256→128→64)")
print(f"Parameters : {n_params:,}")
print(f"Nodes      : {len(proto.graph.node)}")
print(f"File size  : {os.path.getsize(MODEL_PATH)/1024:.1f} KB")

## Exercise 1 — Profiling ORT Sessions

Enable ORT's built-in profiler to get per-node execution times.
The profile is written as a Chrome-trace JSON file.

Open the resulting `.json` in `chrome://tracing` or
[Perfetto UI](https://ui.perfetto.dev/) for a visual timeline.

In [ ]:
so = ort.SessionOptions()
so.enable_profiling = True
so.profile_file_prefix = os.path.join(WORK_DIR, "profile")
so.log_severity_level = 3
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

sess = ort.InferenceSession(MODEL_PATH, so, providers=["CPUExecutionProvider"])

x = np.random.randn(256, 128).astype(np.float32)
# Warmup
for _ in range(30):
    sess.run(None, {"X": x})
# Measured runs
for _ in range(100):
    sess.run(None, {"X": x})

profile_path = sess.end_profiling()
print(f"Profile written: {profile_path}")

with open(profile_path) as f:
    events = json.load(f)

# Aggregate by node/op
from collections import defaultdict
node_times = defaultdict(list)
for e in events:
    if isinstance(e, dict) and e.get("cat") == "Node" and "dur" in e:
        node_times[e["name"]].append(e["dur"])

if node_times:
    total_us = sum(sum(v) for v in node_times.values())
    print(f"\n{'Node':<24} {'Calls':>6} {'Total (µs)':>12} {'Avg (µs)':>10} {'% Total':>8}")
    print("─" * 64)
    for name, durs in sorted(node_times.items(), key=lambda x: -sum(x[1])):
        t = sum(durs)
        pct = 100 * t / total_us if total_us else 0
        print(f"{name:<24} {len(durs):>6} {t:>10.1f} {t/len(durs):>10.2f} {pct:>7.1f}%")
else:
    print("No node-level events found; profile may use a different event schema.")
    cats = set(e.get("cat", "?") for e in events if isinstance(e, dict))
    print(f"  Available categories: {cats}")

## Exercise 2 — Thread Count Sweep

ORT exposes two threading knobs:

- **`intra_op_num_threads`** — threads *within* a single operator (e.g., parallelising a MatMul).
- **`inter_op_num_threads`** — threads *across* independent operators (pipeline-level parallelism).

More threads ≠ always faster.  Cache contention and context-switch overhead
can hurt beyond a sweet spot.

In [ ]:
cpu_count = os.cpu_count() or 4
intra_values = [1, 2, max(1, cpu_count // 2), cpu_count]
# Deduplicate and sort
intra_values = sorted(set(intra_values))
inter_values = [1, 2]

x = np.random.randn(256, 128).astype(np.float32)

print(f"{'intra':>6} {'inter':>6} {'median (ms)':>12} {'p95 (ms)':>12} {'mean (ms)':>12}")
print("─" * 52)

best_config = None
best_median = float("inf")

for intra in intra_values:
    for inter in inter_values:
        so = ort.SessionOptions()
        so.log_severity_level = 3
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        so.intra_op_num_threads = intra
        so.inter_op_num_threads = inter

        sess = ort.InferenceSession(MODEL_PATH, so, providers=["CPUExecutionProvider"])

        for _ in range(50):
            sess.run(None, {"X": x})

        times = []
        for _ in range(200):
            t0 = time.perf_counter()
            sess.run(None, {"X": x})
            times.append(time.perf_counter() - t0)

        arr = np.array(times) * 1000
        med = np.median(arr)
        print(f"{intra:>6} {inter:>6} {med:>10.4f} {np.percentile(arr, 95):>10.4f} {arr.mean():>10.4f}")

        if med < best_median:
            best_median = med
            best_config = (intra, inter)

assert best_median < 100, f"Median too high: {best_median} ms"
print(f"\nBest config: intra={best_config[0]}, inter={best_config[1]} → {best_median:.4f} ms")

## Exercise 3 — Graph Optimisation Level Comparison

ORT applies graph transformations before execution:

| Level | Transformations |
|-------|----------------|
| `ORT_DISABLE_ALL` | None — raw graph |
| `ORT_ENABLE_BASIC` | Constant folding, redundant node elimination |
| `ORT_ENABLE_EXTENDED` | + operator fusion (e.g., MatMul+Add → Gemm) |
| `ORT_ENABLE_ALL` | + layout optimisations, cross-node analysis |

Higher levels generally produce faster graphs but take longer to initialise.

In [ ]:
opt_levels = [
    ("DISABLED",  ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
    ("BASIC",     ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
    ("EXTENDED",  ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
    ("ALL",       ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
]

x = np.random.randn(256, 128).astype(np.float32)

print(f"{'Level':<12} {'Init (ms)':>10} {'Median (ms)':>12} {'p95 (ms)':>10} {'Speedup':>10}")
print("─" * 58)

baseline_median = None

for name, level in opt_levels:
    so = ort.SessionOptions()
    so.log_severity_level = 3
    so.graph_optimization_level = level
    so.intra_op_num_threads = best_config[0]
    so.inter_op_num_threads = best_config[1]

    t_init = time.perf_counter()
    sess = ort.InferenceSession(MODEL_PATH, so, providers=["CPUExecutionProvider"])
    init_ms = (time.perf_counter() - t_init) * 1000

    for _ in range(50):
        sess.run(None, {"X": x})

    times = []
    for _ in range(300):
        t0 = time.perf_counter()
        sess.run(None, {"X": x})
        times.append(time.perf_counter() - t0)

    arr = np.array(times) * 1000
    med = np.median(arr)

    if baseline_median is None:
        baseline_median = med
    speedup = baseline_median / med

    print(f"{name:<12} {init_ms:>8.2f} {med:>10.4f} {np.percentile(arr, 95):>10.4f} {speedup:>9.2f}x")

print("\n(Speedup relative to DISABLED)")

## Exercise 4 — Memory Arena Configuration

ORT uses a **memory arena** (pool allocator) to avoid per-tensor `malloc`/`free`.

| Option | Default | Effect |
|--------|---------|--------|
| `enable_cpu_mem_arena` | True | Use arena allocator |
| `enable_mem_pattern` | True | Pre-compute allocation pattern |

Disabling these can reduce peak memory but increases allocation overhead.

In [ ]:
arena_configs = [
    ("Arena ON + Pattern ON",   True,  True),
    ("Arena ON + Pattern OFF",  True,  False),
    ("Arena OFF + Pattern ON",  False, True),
    ("Arena OFF + Pattern OFF", False, False),
]

x = np.random.randn(512, 128).astype(np.float32)

print(f"{'Config':<30} {'Median (ms)':>12} {'p95 (ms)':>10}")
print("─" * 56)

for label, arena, pattern in arena_configs:
    so = ort.SessionOptions()
    so.log_severity_level = 3
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = best_config[0]
    so.inter_op_num_threads = best_config[1]
    so.enable_cpu_mem_arena = arena
    so.enable_mem_pattern = pattern

    sess = ort.InferenceSession(MODEL_PATH, so, providers=["CPUExecutionProvider"])

    for _ in range(50):
        sess.run(None, {"X": x})

    times = []
    for _ in range(200):
        t0 = time.perf_counter()
        sess.run(None, {"X": x})
        times.append(time.perf_counter() - t0)

    arr = np.array(times) * 1000
    print(f"{label:<30} {np.median(arr):>10.4f} {np.percentile(arr, 95):>10.4f}")

print("\nArena + pattern is typically fastest for repeated inference.")

## Exercise 5 — Batch Size vs Latency Analysis

Increasing batch size amortises per-call overhead but increases memory and
total latency.  We measure:

- **Total latency**: wall-clock time for one `sess.run` call
- **Per-sample latency**: $t_{\text{total}} / N$
- **Throughput**: $N / t_{\text{total}}$ (samples/second)

In [ ]:
so = ort.SessionOptions()
so.log_severity_level = 3
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads = best_config[0]
so.inter_op_num_threads = best_config[1]

sess = ort.InferenceSession(MODEL_PATH, so, providers=["CPUExecutionProvider"])

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]

print(f"{'Batch':>6} {'Total (ms)':>12} {'Per-sample (µs)':>16} {'Throughput':>14}")
print("─" * 52)

results = []
for bs in batch_sizes:
    x = np.random.randn(bs, 128).astype(np.float32)

    for _ in range(30):
        sess.run(None, {"X": x})

    times = []
    for _ in range(150):
        t0 = time.perf_counter()
        sess.run(None, {"X": x})
        times.append(time.perf_counter() - t0)

    med_s = np.median(times)
    total_ms = med_s * 1000
    per_sample_us = total_ms * 1000 / bs
    throughput = bs / med_s

    results.append((bs, total_ms, per_sample_us, throughput))
    print(f"{bs:>6} {total_ms:>10.4f} {per_sample_us:>14.2f} {throughput:>11.0f} samp/s")

# Find the sweet spot
best_tp = max(results, key=lambda r: r[3])
assert best_tp[3] > 0, "Throughput must be positive"
print(f"\nPeak throughput: {best_tp[3]:.0f} samples/s at batch={best_tp[0]}")

## Exercise 6 — Full Benchmark with Percentile Statistics

Production SLAs are stated in percentiles.  We collect enough samples
for stable p50/p95/p99 estimates.

$$\text{p}X = \inf\{t : F(t) \ge X/100\}$$

where $F$ is the empirical CDF of latencies.

In [ ]:
def full_benchmark(sess, feed, warmup=100, iters=500):
    for _ in range(warmup):
        sess.run(None, feed)

    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        sess.run(None, feed)
        times.append(time.perf_counter() - t0)

    arr = np.array(times) * 1000  # ms
    return {
        "min":    arr.min(),
        "p50":    np.percentile(arr, 50),
        "p90":    np.percentile(arr, 90),
        "p95":    np.percentile(arr, 95),
        "p99":    np.percentile(arr, 99),
        "max":    arr.max(),
        "mean":   arr.mean(),
        "std":    arr.std(),
        "iters":  iters,
    }


so = ort.SessionOptions()
so.log_severity_level = 3
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads = best_config[0]
sess = ort.InferenceSession(MODEL_PATH, so, providers=["CPUExecutionProvider"])

for bs in [1, 32, 256]:
    x = np.random.randn(bs, 128).astype(np.float32)
    stats = full_benchmark(sess, {"X": x})

    print(f"\nBatch={bs} ({stats['iters']} iterations):")
    print(f"  min   : {stats['min']:.4f} ms")
    print(f"  p50   : {stats['p50']:.4f} ms")
    print(f"  p90   : {stats['p90']:.4f} ms")
    print(f"  p95   : {stats['p95']:.4f} ms")
    print(f"  p99   : {stats['p99']:.4f} ms")
    print(f"  max   : {stats['max']:.4f} ms")
    print(f"  mean  : {stats['mean']:.4f} ms")
    print(f"  std   : {stats['std']:.4f} ms")
    print(f"  CoV   : {stats['std']/stats['mean']*100:.1f}%")

## Exercise 7 — Model Warm-Up Best Practices

The first few inference calls are slower due to:
1. **Lazy kernel compilation** — ORT JIT-compiles certain ops.
2. **Memory allocation** — arena needs to grow to fit the workload.
3. **CPU cache** — weights aren't in cache yet.

We measure how many calls are needed before latency stabilises.

In [ ]:
so = ort.SessionOptions()
so.log_severity_level = 3
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads = best_config[0]

# Create a FRESH session
sess = ort.InferenceSession(MODEL_PATH, so, providers=["CPUExecutionProvider"])

x = np.random.randn(64, 128).astype(np.float32)

# Record per-call latency for the first 100 calls
all_times = []
for i in range(100):
    t0 = time.perf_counter()
    sess.run(None, {"X": x})
    all_times.append((time.perf_counter() - t0) * 1000)

arr = np.array(all_times)

# Find stabilisation point: when running median stops decreasing significantly
window = 5
running_med = [np.median(arr[max(0, i-window):i+1]) for i in range(len(arr))]
steady_state = np.median(arr[50:])
threshold = steady_state * 1.1  # within 10% of steady state

warmup_done = 0
for i, m in enumerate(running_med):
    if m <= threshold:
        warmup_done = i
        break

print(f"Call-by-call latency (first 20 calls):")
for i in range(20):
    bar = '█' * int(arr[i] / steady_state * 10)
    marker = " ← cold" if i < warmup_done else ""
    print(f"  [{i:3d}] {arr[i]:8.4f} ms  {bar}{marker}")

print(f"\nSteady-state median (calls 50-99): {steady_state:.4f} ms")
print(f"Cold-start call #0              : {arr[0]:.4f} ms")
print(f"Cold-start overhead             : {arr[0] / steady_state:.1f}x")
print(f"Warm-up calls needed            : ~{warmup_done}")
print(f"\nBest practice: run {max(20, warmup_done * 2)} warmup iterations before serving.")

## Exercise 8 — Challenge: Build an Auto-Tuner

Create an `AutoTuner` class that performs a grid search over:
- `intra_op_num_threads`: $[1, 2, \lfloor C/2 \rfloor, C]$
- `inter_op_num_threads`: $[1, 2]$
- `graph_optimization_level`: all four levels
- `enable_cpu_mem_arena`: $[\text{True}, \text{False}]$

The tuner measures median latency for each config and returns the
optimal `SessionOptions`.

In [ ]:
class AutoTuner:
    OPT_LEVELS = [
        ("DISABLED",  ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
        ("BASIC",     ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
        ("EXTENDED",  ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
        ("ALL",       ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
    ]

    def __init__(self, model_path, sample_input):
        self.model_path = model_path
        self.sample_input = sample_input
        self.results = []

    def _benchmark(self, so, warmup=30, iters=100):
        so.log_severity_level = 3
        sess = ort.InferenceSession(
            self.model_path, so, providers=["CPUExecutionProvider"]
        )
        in_name = sess.get_inputs()[0].name
        feed = {in_name: self.sample_input}

        for _ in range(warmup):
            sess.run(None, feed)

        times = []
        for _ in range(iters):
            t0 = time.perf_counter()
            sess.run(None, feed)
            times.append(time.perf_counter() - t0)

        arr = np.array(times) * 1000
        return np.median(arr), np.percentile(arr, 95)

    def tune(self, intra_values=None, inter_values=None, opt_levels=None,
             arena_values=None):
        cpu_c = os.cpu_count() or 4
        if intra_values is None:
            intra_values = sorted(set([1, 2, max(1, cpu_c // 2), cpu_c]))
        if inter_values is None:
            inter_values = [1, 2]
        if opt_levels is None:
            opt_levels = self.OPT_LEVELS
        if arena_values is None:
            arena_values = [True, False]

        total = len(intra_values) * len(inter_values) * len(opt_levels) * len(arena_values)
        print(f"Auto-tuner: {total} configurations to evaluate...\n")

        self.results = []
        for opt_name, opt_level in opt_levels:
            for intra in intra_values:
                for inter in inter_values:
                    for arena in arena_values:
                        so = ort.SessionOptions()
                        so.graph_optimization_level = opt_level
                        so.intra_op_num_threads = intra
                        so.inter_op_num_threads = inter
                        so.enable_cpu_mem_arena = arena

                        med, p95 = self._benchmark(so)
                        config = {
                            "opt_level": opt_name,
                            "intra": intra,
                            "inter": inter,
                            "arena": arena,
                            "median_ms": med,
                            "p95_ms": p95,
                        }
                        self.results.append(config)

        self.results.sort(key=lambda r: r["median_ms"])
        return self.results[0]

    def report(self, top_n=10):
        print(f"\n{'Rank':>4} {'Opt Level':<10} {'intra':>6} {'inter':>6} {'Arena':>6} {'Med (ms)':>10} {'p95 (ms)':>10}")
        print("─" * 58)
        for i, r in enumerate(self.results[:top_n]):
            print(f"{i+1:>4} {r['opt_level']:<10} {r['intra']:>6} {r['inter']:>6} "
                  f"{str(r['arena']):>6} {r['median_ms']:>8.4f} {r['p95_ms']:>10.4f}")

    def make_optimal_session(self):
        if not self.results:
            raise RuntimeError("Run tune() first")
        best = self.results[0]
        so = ort.SessionOptions()
        so.log_severity_level = 3
        for name, lvl in self.OPT_LEVELS:
            if name == best["opt_level"]:
                so.graph_optimization_level = lvl
        so.intra_op_num_threads = best["intra"]
        so.inter_op_num_threads = best["inter"]
        so.enable_cpu_mem_arena = best["arena"]
        return ort.InferenceSession(
            self.model_path, so, providers=["CPUExecutionProvider"]
        )


# Run the auto-tuner (reduced search space for speed)
x_tune = np.random.randn(128, 128).astype(np.float32)
tuner = AutoTuner(MODEL_PATH, x_tune)

best = tuner.tune(
    opt_levels=[
        ("BASIC", ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
        ("ALL",   ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
    ],
    arena_values=[True],
)

print(f"\nOptimal configuration:")
for k, v in best.items():
    print(f"  {k}: {v}")

tuner.report()

# Create the optimal session
optimal_sess = tuner.make_optimal_session()
out = optimal_sess.run(None, {"X": x_tune})
assert out[0].shape == (128, 64), f"Unexpected shape: {out[0].shape}"
print(f"\nOptimal session created — output shape: {out[0].shape} ✓")

## Summary

| Skill | Key Takeaway |
|-------|--------------|
| Profiling | `enable_profiling` produces Chrome-trace JSON; find the hottest nodes |
| Thread tuning | More threads help up to a point; benchmark to find the sweet spot |
| Optimisation levels | ALL is usually best for inference; DISABLED for debugging |
| Memory arena | Arena + memory pattern reduces allocation overhead for repeated calls |
| Batch size | Larger batches amortise overhead; throughput plateaus at some point |
| Percentile stats | Report p50/p95/p99 for production SLAs, not just mean |
| Warm-up | First ~10–30 calls are slower; always warm up before serving |
| Auto-tuning | Grid search over configs finds the optimal combination automatically |